# KITTI Vehicle Tracking — Kaggle Notebook

Run the full YOLO + ByteTrack pipeline on Kaggle with GPU.

**Prerequisites:**  
1. Add KITTI dataset to Kaggle (via `klemenko/kitti-dataset` or upload manually)
2. Enable GPU accelerator in Notebook settings

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    return result

run('pip install -q ultralytics==8.2.27 supervision==0.21.0 motmetrics==1.4.0 rich')
print('Dependencies installed')

In [ ]:
# ── 2. Clone / setup repo ─────────────────────────────────────────────────────
import os, sys
from pathlib import Path

# If running locally from the repo, skip cloning
REPO_DIR = Path('/kaggle/working/kitti_tracker')

if not REPO_DIR.exists():
    # Uncomment to clone your fork:
    # run(f'git clone https://github.com/YOUR_USERNAME/kitti_tracker {REPO_DIR}')
    
    # Or copy from uploaded dataset
    run(f'cp -r /kaggle/input/kitti-tracker-src {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── 3. Locate KITTI data ──────────────────────────────────────────────────────
import os
from pathlib import Path

# Kaggle KITTI dataset paths (klemenko/kitti-dataset)
KAGGLE_KITTI = Path('/kaggle/input/kitti-dataset')

# Check what's available
if KAGGLE_KITTI.exists():
    for p in sorted(KAGGLE_KITTI.rglob('*'))[:20]:
        print(p)
else:
    print('KITTI dataset not found at', KAGGLE_KITTI)
    print('Add it via: Add data → klemenko/kitti-dataset')

In [ ]:
# ── 4. Symlink data ───────────────────────────────────────────────────────────
import os
from pathlib import Path

data_dir = Path('data/kitti')
data_dir.mkdir(parents=True, exist_ok=True)

# Adjust these paths based on the actual Kaggle dataset structure
# Common structures:
#   /kaggle/input/kitti-dataset/training/
#   /kaggle/input/kitti-object-detection-dataset/training/

kitti_input = Path('/kaggle/input/kitti-dataset')
training_src = kitti_input / 'training'

if training_src.exists():
    target = data_dir / 'training'
    if not target.exists():
        os.symlink(training_src, target)
    print(f'Linked {training_src} → {target}')
    
    # Verify
    img_dir = target / 'image_02'
    if img_dir.exists():
        seqs = sorted(img_dir.iterdir())
        print(f'Found {len(seqs)} sequences')
    else:
        print('image_02 not found — check dataset structure')
else:
    print(f'Source not found: {training_src}')
    print('Available inputs:', list(kitti_input.iterdir()) if kitti_input.exists() else 'N/A')

In [ ]:
# ── 5. Check GPU ──────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 6. Run pipeline ───────────────────────────────────────────────────────────
import yaml
from pathlib import Path
import sys
sys.path.insert(0, '.')

from src.data_loader import KITTITrackingDataset
from src.detector import KITTIDetector
from src.tracker import KITTITracker
from src.pipeline import run_all_sequences

cfg = yaml.safe_load(open('configs/config.yaml'))

# Components
detector = KITTIDetector(
    model_path='yolov8m.pt',      # downloads automatically
    conf_threshold=0.25,
    iou_threshold=0.45,
    device='cuda',
    half_precision=True,
    img_size=1280,
)

tracker = KITTITracker(
    track_thresh=0.25,
    match_thresh=0.8,
    track_buffer=30,
    frame_rate=10,
)

dataset = KITTITrackingDataset(
    kitti_root='data/kitti',
    split='training',
    sequences=[0, 1, 2, 3, 4],    # adjust as needed
    allowed_classes=['Car', 'Pedestrian', 'Cyclist'],
    min_height=25,
    max_occlusion=2,
    max_truncation=0.5,
)

print(f'Sequences: {dataset.sequence_ids}')

# Run
results = run_all_sequences(
    dataset=dataset,
    detector=detector,
    tracker=tracker,
    output_dir=Path('outputs'),
    eval_sequences=dataset.sequence_ids,
    fps=10.0,
    iou_threshold=0.5,
    classes=['Car', 'Pedestrian', 'Cyclist'],
    visualize=True,
    demo_seq_id=0,
)

print('\nResults:')
print(results.to_string(index=False))

In [ ]:
# ── 7. Display demo video ─────────────────────────────────────────────────────
from IPython.display import Video
from pathlib import Path

demo = Path('outputs/videos/tracking_demo.mp4')
if demo.exists():
    display(Video(str(demo), embed=True, width=1000))
else:
    print('Video not found:', demo)
    print('Available outputs:', list(Path('outputs').rglob('*.mp4')))

In [ ]:
# ── 8. MOTA results table ─────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

csv = Path('outputs/mota_results.csv')
if csv.exists():
    df = pd.read_csv(csv)
    print('\nMOTA Results:')
    display(df.style.highlight_max(
        subset=['mota','idf1','recall','precision'],
        color='lightgreen'
    ).highlight_min(
        subset=['num_switches','num_false_positives','num_misses'],
        color='lightgreen'
    ).format({
        c: '{:.2f}' for c in df.select_dtypes('float').columns
    }))
else:
    print('Results CSV not found')

In [ ]:
# ── 9. Copy outputs for download ──────────────────────────────────────────────
import shutil
from pathlib import Path

out_kaggle = Path('/kaggle/working/outputs')
out_kaggle.mkdir(exist_ok=True)

src_outputs = Path('outputs')
if src_outputs.exists():
    shutil.copytree(src_outputs, out_kaggle, dirs_exist_ok=True)
    print('Outputs copied to /kaggle/working/outputs')
    for f in sorted(out_kaggle.rglob('*')):
        if f.is_file():
            print(f'  {f.relative_to(out_kaggle)}  ({f.stat().st_size/1e6:.1f} MB)')